# Sentiment analysis

### Loading dataset

In [1]:
import pandas as pd

In [3]:
data = pd.read_csv("/kaggle/input/datasets/jp797498e/twitter-entity-sentiment-analysis/twitter_training.csv")

In [3]:
data.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [4]:
data.shape

(74681, 4)

In [5]:
data=data[:5000]

In [6]:
data.isnull().value_counts()

2401   Borderlands  Positive  im getting on borderlands and i will murder you all ,
False  False        False     False                                                    4952
                              True                                                       48
Name: count, dtype: int64

In [7]:
data = data.dropna()

In [8]:
data.isnull().value_counts()

2401   Borderlands  Positive  im getting on borderlands and i will murder you all ,
False  False        False     False                                                    4952
Name: count, dtype: int64

In [9]:
X_unprocessed = data["im getting on borderlands and i will murder you all ,"]

In [10]:
X_unprocessed

0       I am coming to the borders and I will kill you...
1       im getting on borderlands and i will kill you ...
2       im coming on borderlands and i will murder you...
3       im getting on borderlands 2 and i will murder ...
4       im getting into borderlands and i can murder y...
                              ...                        
4995    @OregonChai What did y ’ all really stop makin...
4996    @OregonChai did y’all stop making the butter<u...
4997    Amazon UK launches the Sherlock Holmes Advent ...
4998    Amazon UK launches the Sherlock Holmes Advent ...
4999    Amazon UK launches Sherlock Holmes Advent Cale...
Name: im getting on borderlands and i will murder you all ,, Length: 4952, dtype: object

In [11]:
X_unprocessed = list(X_unprocessed)
X_unprocessed

['I am coming to the borders and I will kill you all,',
 'im getting on borderlands and i will kill you all,',
 'im coming on borderlands and i will murder you all,',
 'im getting on borderlands 2 and i will murder you me all,',
 'im getting into borderlands and i can murder you all,',
 "So I spent a few hours making something for fun. . . If you don't know I am a HUGE @Borderlands fan and Maya is one of my favorite characters. So I decided to make myself a wallpaper for my PC. . Here is the original image versus the creation I made :) Enjoy! pic.twitter.com/mLsI5wf9Jg",
 "So I spent a couple of hours doing something for fun... If you don't know that I'm a huge @ Borderlands fan and Maya is one of my favorite characters, I decided to make a wallpaper for my PC.. Here's the original picture compared to the creation I made:) Have fun! pic.twitter.com / mLsI5wf9Jg",
 "So I spent a few hours doing something for fun... If you don't know I'm a HUGE @ Borderlands fan and Maya is one of my fav

In [12]:
y_unprocessed = data["Positive"]
y_unprocessed

0       Positive
1       Positive
2       Positive
3       Positive
4       Positive
          ...   
4995     Neutral
4996     Neutral
4997     Neutral
4998     Neutral
4999     Neutral
Name: Positive, Length: 4952, dtype: object

In [18]:
y_unprocessed.value_counts()

Positive
Positive      1926
Neutral       1147
Negative      1046
Irrelevant     833
Name: count, dtype: int64

- Load tokenizer and encoder model 
- using BERT model

In [19]:
from transformers import BertTokenizer, BertModel
import torch

In [20]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').to(device=torch.device('cuda'))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
tokenized_X = tokenizer(X_unprocessed, return_tensors="pt", padding=True, truncation=True)
tokenized_X

{'input_ids': tensor([[  101,  1045,  2572,  ...,     0,     0,     0],
        [  101, 10047,  2893,  ...,     0,     0,     0],
        [  101, 10047,  2746,  ...,     0,     0,     0],
        ...,
        [  101,  9733,  2866,  ...,     0,     0,     0],
        [  101,  9733,  2866,  ...,     0,     0,     0],
        [  101,  9733,  2866,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

- send processed `tokenized_X` data to gpu as our model is loaded on gpu.

In [22]:
tokenized_X.to(device=torch.device('cuda'))

{'input_ids': tensor([[  101,  1045,  2572,  ...,     0,     0,     0],
        [  101, 10047,  2893,  ...,     0,     0,     0],
        [  101, 10047,  2746,  ...,     0,     0,     0],
        ...,
        [  101,  9733,  2866,  ...,     0,     0,     0],
        [  101,  9733,  2866,  ...,     0,     0,     0],
        [  101,  9733,  2866,  ...,     0,     0,     0]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

- lets do the embedding of our processed tokens
- but before that we need to create batches for our dataset

In [23]:
from torch.utils.data import TensorDataset, DataLoader

In [24]:
dataset = TensorDataset(tokenized_X["input_ids"], tokenized_X["attention_mask"])
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

- storing our processed encoded X data into variable named `all_embeddings`

In [25]:
all_embeddings = []


In [26]:
with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask = batch

        #  Move inputs to CUDA BEFORE passing to the model
        input_ids = input_ids.to('cuda')
        attention_mask = attention_mask.to('cuda')

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # If its a hugging face model, you usually want the last hidden state as the embeddings
        embeddings = outputs["last_hidden_state"][:,0,:]


        # Move ONLY the specific tensor to CPU
        all_embeddings.append(embeddings.cpu())

X_embeddings = torch.cat(all_embeddings, dim=0)


In [27]:
X_embeddings

tensor([[ 0.1977, -0.0162,  0.0218,  ..., -0.4599,  0.2951,  0.3930],
        [ 0.0387,  0.3623,  0.1580,  ..., -0.5341,  0.8524,  0.2243],
        [-0.0569, -0.2318, -0.0112,  ..., -0.2283,  0.3926,  0.3825],
        ...,
        [-0.5807, -0.3734,  0.1458,  ..., -0.1708,  0.4344,  0.5682],
        [ 0.3169, -0.2123,  0.3525,  ..., -0.3268,  0.3682,  0.5125],
        [-0.0105,  0.0340,  0.0703,  ..., -0.3281,  0.1652,  0.2758]])

In [28]:
X_embeddings.shape

torch.Size([4952, 768])

- Now lets work with `y_unprocessed` data

In [31]:
y_unprocessed

0       Positive
1       Positive
2       Positive
3       Positive
4       Positive
          ...   
4995     Neutral
4996     Neutral
4997     Neutral
4998     Neutral
4999     Neutral
Name: Positive, Length: 4952, dtype: object

In [32]:
y_unprocessed.shape

(4952,)

In [33]:
# using scikit-learns label encoder
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_processed = le.fit_transform(y_unprocessed)
y_processed

array([3, 3, 3, ..., 2, 2, 2])

In [34]:
X = torch.tensor(X_embeddings, dtype=torch.float, device=torch.device('cuda'))
y = torch.tensor(y_processed, dtype=torch.long, device=torch.device('cuda'))

/tmp/ipykernel_105/127478103.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X_embeddings, dtype=torch.float, device=torch.device('cuda'))


In [35]:
y

tensor([3, 3, 3,  ..., 2, 2, 2], device='cuda:0')

# Data train test split

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([3961, 768]),
 torch.Size([991, 768]),
 torch.Size([3961]),
 torch.Size([991]))

# Model Training

In [38]:
import torch.nn as nn

In [55]:
class SentimentModel(nn.Module):
    def __init__(self, input_dim=768, hidden_1=384, hidden_2=192, hidden_3=96, hidden_4=48, hidden_5=12, hidden_6=6, output_dim=4):
        super(SentimentModel, self).__init__()
        
        # layers
        
        self.fc1 = nn.Linear(input_dim, hidden_1) # input layer
        
        # hidden layer start
        self.fc2 = nn.Linear(hidden_1, hidden_2) # hidden layer 1
        self.relu1 = nn.ReLU()
        
        self.fc3 = nn.Linear(hidden_2, hidden_3) # hidden layer 2
        self.relu2 = nn.ReLU()
        
        self.fc4 = nn.Linear(hidden_3, hidden_4) # hidden layer 3 
        self.relu3 = nn.ReLU()
        
        self.fc5 = nn.Linear(hidden_4, hidden_5) # hidden layer 4
        self.relu4 = nn.ReLU()
        
        self.fc6 = nn.Linear(hidden_5, hidden_6) # hidden layer 5
        self.relu5 = nn.ReLU()
        # hidden layer end
        
        self.fc7 = nn.Linear(hidden_6, output_dim) # output layer
        
    def forward(self, x):
        x = self.fc1(x)
        
        x = self.relu1(self.fc2(x))
        x = self.relu2(self.fc3(x))
        x = self.relu3(self.fc4(x))
        x = self.relu4(self.fc5(x))
        x = self.relu5(self.fc6(x))
        
        x = self.fc7(x)
        
        return x
        

In [56]:
sentiment_model = SentimentModel()
sentiment_model.to(device=torch.device('cuda'))

SentimentModel(
  (fc1): Linear(in_features=768, out_features=384, bias=True)
  (fc2): Linear(in_features=384, out_features=192, bias=True)
  (relu1): ReLU()
  (fc3): Linear(in_features=192, out_features=96, bias=True)
  (relu2): ReLU()
  (fc4): Linear(in_features=96, out_features=48, bias=True)
  (relu3): ReLU()
  (fc5): Linear(in_features=48, out_features=12, bias=True)
  (relu4): ReLU()
  (fc6): Linear(in_features=12, out_features=6, bias=True)
  (relu5): ReLU()
  (fc7): Linear(in_features=6, out_features=4, bias=True)
)

In [57]:
# create data batches 
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


In [60]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(sentiment_model.parameters(), lr=0.001)

In [61]:
# model training loop 

epoches = 501

for epoch in range(epoches):
    sentiment_model.train()

    out = sentiment_model(X_train)
    loss = loss_function(out, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 ==0:
        print("Epoch: ", epoch, " Loss: ", loss.item())

Epoch:  0  Loss:  1.354526162147522
Epoch:  10  Loss:  1.3378897905349731
Epoch:  20  Loss:  1.333010196685791
Epoch:  30  Loss:  1.3263088464736938
Epoch:  40  Loss:  1.314558744430542
Epoch:  50  Loss:  1.298778772354126
Epoch:  60  Loss:  1.284065842628479
Epoch:  70  Loss:  1.2713371515274048
Epoch:  80  Loss:  1.253223180770874
Epoch:  90  Loss:  1.2519245147705078
Epoch:  100  Loss:  1.2205899953842163
Epoch:  110  Loss:  1.209118366241455
Epoch:  120  Loss:  1.2000776529312134
Epoch:  130  Loss:  1.1838464736938477
Epoch:  140  Loss:  1.1796058416366577
Epoch:  150  Loss:  1.1555744409561157
Epoch:  160  Loss:  1.13692045211792
Epoch:  170  Loss:  1.147875189781189
Epoch:  180  Loss:  1.1430256366729736
Epoch:  190  Loss:  1.1054925918579102
Epoch:  200  Loss:  1.0825425386428833
Epoch:  210  Loss:  1.0851123332977295
Epoch:  220  Loss:  1.040624737739563
Epoch:  230  Loss:  1.0232170820236206
Epoch:  240  Loss:  1.0100206136703491
Epoch:  250  Loss:  1.0013270378112793
Epoch:  

- Lets predict sentiment of a example text

In [75]:
text = ["You are so fucked up person"]

In [76]:
tokenized_text = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
tokenized_text = tokenized_text.to(device=torch.device("cuda"))
tokenized_text

{'input_ids': tensor([[  101,  2017,  2024,  2061, 21746,  2039,  2711,   102]],
       device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [77]:
embedded_text = model(**tokenized_text)
embedded_text

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.1174,  0.2629, -0.5414,  ..., -0.5146,  0.3503,  0.2373],
         [ 0.0570, -0.0580, -0.4426,  ...,  0.1087,  1.1869, -0.3512],
         [ 0.8722,  0.0934, -0.3761,  ..., -0.1744,  0.7663, -0.1675],
         ...,
         [ 0.5239,  0.3146, -0.5485,  ..., -0.6218,  0.3865, -0.5152],
         [-0.1331,  0.0115, -0.3314,  ...,  0.2922,  0.2628, -0.6514],
         [ 0.8309, -0.0747, -0.2740,  ..., -0.3405, -0.5661, -0.0999]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-0.9028, -0.5223, -0.9102,  0.9080,  0.7653, -0.3104,  0.9094,  0.3632,
         -0.8909, -1.0000, -0.6076,  0.9541,  0.9864,  0.6069,  0.9617, -0.8254,
         -0.5637, -0.6803,  0.3872, -0.6690,  0.7623,  1.0000, -0.0054,  0.3245,
          0.4662,  0.9937, -0.8065,  0.9543,  0.9638,  0.6721, -0.8595,  0.2431,
         -0.9920, -0.3353, -0.9225, -0.9953,  0.5515, -0.7197,  0.0193, -0.0445,
         -0.941

In [78]:
embedded_text_processed = embedded_text["last_hidden_state"][:, 0, :]
embedded_text_processed.shape

torch.Size([1, 768])

In [79]:
# Lets predict 
out = sentiment_model(embedded_text_processed)
out.argmax(dim=1)

tensor([3], device='cuda:0')

In [80]:
out = out.argmax(dim=1)
out

tensor([3], device='cuda:0')

In [81]:
y_processed

array([3, 3, 3, ..., 2, 2, 2])

In [82]:
le

LabelEncoder()

In [83]:
out_new = out.detach().cpu().numpy()
out_new

array([3])

In [84]:
le.inverse_transform(out_new)

array(['Positive'], dtype=object)